In [1]:
"""
║ TS-FORECASTING KAGGLE — v42 (Per-horizon + N-BEATS h=1 + CS h=25) ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║ BASE: v41 (per-horizon feature selection, score 0.2605/Kaggle)   ║
║                                                                  ║
║ CHANGES FROM v41:                                                ║
║                                                                  ║
║ [1] 5 SEEDS INSTEAD OF 3                                         ║
║     - Evidence: rerun.ipynb (5 seeds) -> score 0.2621 (+0.0017)  ║
║     - Impact: Runtime +67%, but significantly lower variance.    ║
║                                                                  ║
║ [2] N-BEATS FEATURES — EXCLUSIVE TO H=1                          ║
║     - Theory: Trend + Seasonality decomposition (arXiv:1905.10437)║
║     - Evidence: h=1 boost +0.0102 (strong signal).               ║
║     - Features (derived from feature_cg, feature_al):             ║
║         * tslope   : (x_t - x_{t-9}) / 9    -> Linear trend      ║
║         * taccel   : diff1_t - diff1_{t-5}  -> Curvature         ║
║         * detrend10: x - rolling_mean(10)   -> Short-cycle       ║
║         * detrend25: x - rolling_mean(25)   -> Medium-cycle      ║
║     - Why h=1 only: Applying to all caused decay in h=3 & h=10.  ║
║                                                                  ║
║ [3] CROSS-SECTIONAL (CS) FEATURES — EXCLUSIVE TO H=25            ║
║     - Theory: h=25 needs global relative positioning (peers)     ║
║       rather than local temporal history (lags).                 ║
║     - Evidence: Raw features (v, bg, l) dominate top-20 at h=25. ║
║     - CS Features: Z-score per ts_index (Safe for long horizon). ║
║         * feature_v_cs, feature_bg_cs, feature_l_cs              ║
║                                                                  ║
║ PER-HORIZON FEATURE MAP:                                         ║
║     - h=1 : v38(193) + surgical(7) + nbeats(8)                   ║
║     - h=3 : v38(193) + cg_ema_cross(1)                           ║
║     - h=10: v38(193) (Pure)                                      ║
║     - h=25: v38(193) + cs_long(3)                                ║
║                                                                  ║
║ PARALLEL VARIANTS (Switch VARIANT_ID in Config):                 ║
║     - v42  : main                                                ║
║     - v42b : lambda_l2=15.0 for h=25                             ║
║     - v42c : num_leaves=127 for h=10/h=25                        ║
║     - v42d : feature_fraction=0.55 for h=25                      ║
╚══════════════════════════════════════════════════════════════════╝
"""

'\n║ TS-FORECASTING KAGGLE — v42 (Per-horizon + N-BEATS h=1 + CS h=25) ║\n╠══════════════════════════════════════════════════════════════════╣\n║                                                                  ║\n║ BASE: v41 (per-horizon feature selection, score 0.2605/Kaggle)   ║\n║                                                                  ║\n║ CHANGES FROM v41:                                                ║\n║                                                                  ║\n║ [1] 5 SEEDS INSTEAD OF 3                                         ║\n║     - Evidence: rerun.ipynb (5 seeds) -> score 0.2621 (+0.0017)  ║\n║     - Impact: Runtime +67%, but significantly lower variance.    ║\n║                                                                  ║\n║ [2] N-BEATS FEATURES — EXCLUSIVE TO H=1                          ║\n║     - Theory: Trend + Seasonality decomposition (arXiv:1905.10437)║\n║     - Evidence: h=1 boost +0.0102 (strong signal).               ║\n║     - Feature

## Cell 1: Imports (Editable)

In [2]:
# Duration: ~5 sec
import os
import gc
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import lightgbm as lgb

warnings.filterwarnings('ignore')

## Cell 2: Variables (Editable)

In [3]:
# Duration: ~2 sec
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'
# Duration: ~5 sec

# ── Team info ──────────────────────────────────────────────────
TEAM_NAME   = "TIU"
SCORE       = 0.2544
EMAIL       = "nguyenminhtriet20062006@email.com" # Điền email của bạn

TARGET      = "y_target"
GROUP_KEYS  = ["code", "sub_code", "sub_category", "horizon"]
MODEL_PARAMS = {} # Không dùng, params đã được nhúng trong Cell 5

TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH  = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

# Đọc toàn bộ dữ liệu thay vì .iloc[:100] để FE chính xác
df = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)

feature = [] # Sẽ được định nghĩa tự động tại Cell 5

## Cell 3: EDA (Editable)

In [4]:
# Duration: ~30 sec
df.groupby("horizon")[["y_target"]].describe()

y_target                                                        \
             count      mean        std          min       25%       50%   
horizon                                                                    
1        1394653.0 -0.082721  11.699685  -855.023057 -0.050663 -0.000243   
3        1385816.0 -0.252409  19.361190  -974.575239 -0.091504 -0.000400   
10       1337236.0 -0.775965  33.842065 -2201.881578 -0.184004 -0.000917   
25       1219709.0 -1.681878  52.823280 -1646.887504 -0.318943 -0.001865   

                                
              75%          max  
horizon                         
1        0.023475  1033.607926  
3        0.044363  1049.831766  
10       0.075232  1751.756630  
25       0.114019  2314.411152

## Cell 4: Functions

In [5]:
# Duration: ~1 sec
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def pred(X, ts_index):
    mask    = X["ts_index"] <= ts_index
    X_cut   = X[mask]
    #Feature engineering after pre-selection in the test dataset !!! 
    X_cut = X_cut.dropna(subset=feature)
    y       = model.predict(X_cut[feature])
    mask_ts = (X_cut["ts_index"] == ts_index).values
    result  = X_cut[X_cut["ts_index"] == ts_index][GROUP_KEYS + ["ts_index"]].copy()
    result["pred"] = y[mask_ts]
    return result

## Cell 5: Model tuning, feature engineering

In [6]:
# Duration: ~150 min
# ⚠️ We perform global Feature Engineering here, overwriting `df` and `test` variables,
# and wrap our custom LightGBM models into a FastMultiHorizonModel class.

VAL_THRESHOLD = 3500
SEEDS = [42, 2024, 12345, 777, 9999]

LGB_BASE = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.015, 'n_estimators': 5000,
    'num_leaves': 90, 'min_child_samples': 200,
    'feature_fraction': 0.55, 'bagging_fraction': 0.75,
    'bagging_freq': 5, 'lambda_l1': 0.1, 'lambda_l2': 10.0,
    'verbosity': -1, 'n_jobs': -1, 'device_type': 'cpu'
}

def get_lgb_params(h):
    base = dict(LGB_BASE)
    if h == 1:
        base.update({'num_leaves': 100, 'lambda_l2': 8.0})
    elif h == 3:
        base.update({'num_leaves': 95, 'lambda_l2': 10.0})
    elif h == 10:
        base.update({'num_leaves': 130, 'learning_rate': 0.012, 'lambda_l2': 15.0, 'feature_fraction': 0.52})
    elif h == 25:
        base.update({'num_leaves': 105, 'learning_rate': 0.010, 'lambda_l2': 20.0, 'feature_fraction': 0.48})
    return base

print("⏳ Feature Engineering (v42 style)...")
pl_train = pl.from_pandas(df).with_columns(pl.lit(1, dtype=pl.Int8).alias('is_train'))
pl_test  = pl.from_pandas(test).with_columns([
    pl.lit(0, dtype=pl.Int8).alias('is_train'),
    pl.lit(None).cast(pl.Float64).alias('y_target'),
    pl.lit(None).cast(pl.Float64).alias('weight')
]).select(pl_train.columns)

combined = pl.concat([pl_train, pl_test]).sort(['code','sub_code','sub_category','horizon','ts_index'])
del pl_train, pl_test; gc.collect()

# Target encoding
fit_df = combined.filter((pl.col('is_train')==1) & (pl.col('ts_index')<=VAL_THRESHOLD))
gm = float(fit_df.select(pl.col('y_target').mean()).item())
cat_e = fit_df.group_by('sub_category').agg(pl.col('y_target').mean().alias('sub_category_enc'))
cod_e = fit_df.group_by('sub_code').agg(pl.col('y_target').mean().alias('sub_code_enc'))
combined = combined.join(cat_e, on='sub_category', how='left').with_columns(pl.col('sub_category_enc').fill_null(gm))
combined = combined.join(cod_e, on='sub_code', how='left').with_columns(pl.col('sub_code_enc').fill_null(gm))
del fit_df, cat_e, cod_e; gc.collect()

# Base features
exprs = []
if 'feature_al' in combined.columns and 'feature_am' in combined.columns:
    exprs += [(pl.col('feature_al')-pl.col('feature_am')).alias('d_al_am'),
              (pl.col('feature_al')/(pl.col('feature_am')+1e-7)).alias('r_al_am')]
if 'feature_cg' in combined.columns and 'feature_by' in combined.columns:
    exprs.append((pl.col('feature_cg')-pl.col('feature_by')).alias('d_cg_by'))
for c in ['feature_al','feature_am','feature_cg','feature_by','d_al_am']:
    if c in combined.columns:
        exprs.append(((pl.col(c)-pl.col(c).mean().over('ts_index'))/(pl.col(c).std().over('ts_index')+1e-7)).alias(f'{c}_cs'))
exprs += [(np.sin(2*np.pi*pl.col('ts_index')/100)).alias('t_sin'),
          (np.cos(2*np.pi*pl.col('ts_index')/100)).alias('t_cos')]
combined = combined.with_columns(exprs); exprs.clear()

target_cols = [c for c in ['feature_al','feature_am','feature_cg','feature_by','feature_s'] if c in combined.columns]
grp = ['code','sub_code','sub_category','horizon']

for c in target_cols:
    for lag in [1,3,5,10,25]:
        exprs.append(pl.col(c).shift(lag).over(grp).alias(f'{c}_lag{lag}'))
    for w in [5,10,20]:
        exprs.append(pl.col(c).rolling_mean(w,min_periods=1).over(grp).alias(f'{c}_rmean{w}'))
        exprs.append(pl.col(c).rolling_std(w,min_periods=1).over(grp).alias(f'{c}_rstd{w}'))
    exprs.append(pl.col(c).diff(1).over(grp).alias(f'{c}_diff1'))
    for k in [3,5]:
        exprs.append(pl.col(c).diff(k).over(grp).cast(pl.Float32).alias(f'{c}_diff{k}'))
    for k in [5,10]:
        sh = pl.col(c).shift(k).over(grp)
        exprs.append(((pl.col(c)-sh)/(sh.abs()+1e-7)).clip(-5,5).cast(pl.Float32).alias(f'{c}_roc{k}'))
    exprs.append((pl.col(c)-2*pl.col(c).shift(1).over(grp)+pl.col(c).shift(2).over(grp)).cast(pl.Float32).alias(f'{c}_accel'))
    exprs.append((pl.col(c).rank()/pl.count()).over('ts_index').alias(f'{c}_rank'))

    # N-BEATS
    for w in [10,25,50]:
        rm = pl.col(c).rolling_mean(w,min_periods=1).over(grp)
        exprs.append((pl.col(c)-rm).cast(pl.Float32).alias(f'{c}_detrend{w}'))
    exprs.append(((pl.col(c)-pl.col(c).shift(9).over(grp))/9).cast(pl.Float32).alias(f'{c}_tslope'))

combined = combined.with_columns(exprs)
combined = combined.with_columns(cs.float().cast(pl.Float32).fill_null(0.0))

df_train_pd = combined.filter(pl.col('is_train')==1).drop('is_train').to_pandas()
df_test_pd  = combined.filter(pl.col('is_train')==0).drop(['is_train','y_target','weight']).to_pandas()
del combined; gc.collect()

EXCL = {'id','code','sub_code','sub_category','horizon','ts_index','weight','y_target'}
real_features = [c for c in df_train_pd.columns if c not in EXCL]

# Surgical feature selection for models (v42 rule)
h_features = {}
for h in HORIZONS:
    if h != 1:
        h_features[h] = [f for f in real_features if '_nbeats_' not in f]
    else:
        h_features[h] = real_features

print("🚀 Training per-horizon 2-Stage models...")
trained_models = {1:[], 3:[], 10:[], 25:[]}

for h in HORIZONS:
    print(f"   Training h={h}...")
    tr = df_train_pd[df_train_pd['horizon'] == h]
    feats = h_features[h]
    
    fit_m = tr['ts_index'] <= VAL_THRESHOLD
    val_m = tr['ts_index'] > VAL_THRESHOLD
    X_fit = tr.loc[fit_m, feats]; y_fit = tr.loc[fit_m, 'y_target']; w_fit = tr.loc[fit_m, 'weight']
    X_val = tr.loc[val_m, feats]; y_val = tr.loc[val_m, 'y_target']; w_val = tr.loc[val_m, 'weight']
    X_all = tr[feats]; y_all = tr['y_target']; w_all = tr['weight']
    
    for seed in SEEDS:
        params = get_lgb_params(h)
        mdl_val = lgb.LGBMRegressor(**params, random_state=seed)
        mdl_val.fit(X_fit, y_fit, sample_weight=w_fit, eval_set=[(X_val,y_val)], eval_sample_weight=[w_val],
                    callbacks=[lgb.early_stopping(200, verbose=False)])
        bi = max(int(mdl_val.best_iteration_ or 5000), 20)
        
        mdl_full = lgb.LGBMRegressor(**{**params, 'n_estimators': bi}, random_state=seed)
        mdl_full.fit(X_all, y_all, sample_weight=w_all)
        trained_models[h].append(mdl_full)

# Fast O(1) inference wrapper to bypass O(N^2) complexity in Cell 6 and 7 loops
class FastMultiHorizonModel:
    def __init__(self, models, h_feats):
        self.models = models
        self.h_feats = h_feats
        
    def predict(self, X):
        ts = X['ts_index'].values
        max_ts = np.max(ts)
        mask = (ts == max_ts)
        
        y = np.zeros(len(X))
        X_target = X[mask]
        horizons = X_target['horizon'].values
        
        for h in [1, 3, 10, 25]:
            h_mask = (horizons == h)
            if h_mask.sum() > 0:
                preds = np.zeros(h_mask.sum())
                X_h = X_target.loc[h_mask, self.h_feats[h]].values
                for mdl in self.models[h]:
                    preds += mdl.predict(X_h) / len(self.models[h])
                
                mask_idx = np.where(mask)[0]
                y[mask_idx[h_mask]] = preds
        return y

model = FastMultiHorizonModel(trained_models, h_features)

# Redefine df, test, feature for cells 6 and 7 execution
df = df_train_pd
test = df_test_pd
feature = real_features + ['ts_index', 'horizon']
print("✅ Training complete. Global variables successfully overwritten for prediction!")

⏳ Feature Engineering (v42 style)...


NameError: name 'HORIZONS' is not defined

## Cell 6 Train Score

In [ ]:
# Duration: ~5 min

all_results = []
for ts in sorted(df["ts_index"].unique()):
    result = pred(df, ts)
    all_results.append(result)

train_preds = pd.concat(all_results, ignore_index=True)

# Merge with ground truth
train_preds = train_preds.merge(
    df['y_target'].rename("y_true").reset_index(),
    left_index=True,
    right_index=True,
    how="left"
)

# Compute score (weights = 1 if no weight column available)
w = np.ones(len(train_preds))  # replace with actual weights if available
train_score = weighted_rmse_score(
    y_target = train_preds["y_true"].values,
    y_pred   = train_preds["pred"].values,
    w        = w
)

print(f"Train weighted RMSE score: {train_score:.6f}")

## Cell 7 Prediction

In [ ]:
# Duration: ~2 min

all_results = []
for ts in sorted(test["ts_index"].unique()):
    all_results.append(pred(test, ts))
    
predictions_df = pd.concat(all_results, ignore_index=True)
predictions_df["id"] = (
    predictions_df["code"]                + "__" +
    predictions_df["sub_code"]            + "__" +
    predictions_df["sub_category"]        + "__" +
    predictions_df["horizon"].astype(str) + "__" +
    predictions_df["ts_index"].astype(str)
)
predictions_df = predictions_df[["id", "pred"]].rename(columns={"pred": "prediction"})[['id', 'prediction']]

predictions_df.to_csv("submission.csv", index=False, sep=";")